In [2]:
import pandas as pd
import os
from tqdm.auto import tqdm
from llm_asr_clarification.constants import SAMPLE_MEETINGS
import json
import torch
import torch.nn.functional as F
from rouge_score import rouge_scorer
from transformers import AutoTokenizer
from jiwer import wer
import jiwer

# Define a robust transformation pipeline
# This applies data cleaning steps in order, from top to bottom
JIWER_TRANSFORM = jiwer.Compose([
    jiwer.ToLowerCase(),                # Convert all text to lowercase
    jiwer.RemovePunctuation(),          # Strip characters like commas, periods, question marks
    jiwer.RemoveMultipleSpaces(),       # Turn multi-spaces into a single space
    jiwer.Strip(),                      # Clean up leading/trailing whitespaces
    jiwer.ReduceToListOfListOfWords()   # Format text tokens perfectly for jiwer's internal engine
])


pd.set_option('display.max_colwidth', None)

MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"   # or another causal LM
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
scorer = rouge_scorer.RougeScorer(
    ["rougeL"],
    # use_stemmer=True,
    use_stemmer=False
)

def rouge_l(pred, ref):
    return scorer.score(
        ref,   # reference first
        pred   # prediction second
    )["rougeL"]

def load_df_from_path(AMI_PATH):
    meeting_paths = [entry.path for entry in os.scandir(AMI_PATH)]
        
    data = []
    for meeting_path in tqdm(meeting_paths):
        beam_results = os.path.join(meeting_path, "artifacts", "beam_results.json")
        try:
            with open(beam_results, "r", encoding="utf-8") as f:
                lines = f.read()
            lines = json.loads(lines)
        except Exception as err:
            print(f"couldnt open file {beam_results}")
            continue
    
        # Process lines
        for line in lines:
            for i in range(1,6):
                beam_no = f'beam_{i}'
                beam = line.pop(beam_no)
    
                line[f"{beam_no}_text"] = beam["text"]
                line[f"{beam_no}_asrlogprob"] = beam["asr_avg_log_prob"]
                line[f"{beam_no}_llmlogprob"] = beam["llm_avg_log_prob"]
            line["meeting_name"] = meeting_path.split("/")[-1]

            data.append(line)
            
    # df = pd.DataFrame(lines)
    df = pd.DataFrame(data)
    return df
    
AMI_TRAIN_PATH = '/group/jrwhitehill/amicorpus/train'
AMI_VAL_PATH = '/group/jrwhitehill/amicorpus/validation'

df = load_df_from_path(AMI_TRAIN_PATH)
df_val = load_df_from_path(AMI_VAL_PATH)

print(df.shape)
print(df_val.shape)

  0%|          | 0/136 [00:00<?, ?it/s]

  0%|          | 0/18 [00:00<?, ?it/s]

(88717, 17)
(10590, 17)


In [3]:
df.head(3)

,gt,beam_1_text,beam_1_asrlogprob,beam_1_llmlogprob,beam_2_text,beam_2_asrlogprob,beam_2_llmlogprob,beam_3_text,beam_3_asrlogprob,beam_3_llmlogprob,beam_4_text,beam_4_asrlogprob,beam_4_llmlogprob,beam_5_text,beam_5_asrlogprob,beam_5_llmlogprob,meeting_name
0,Okay.,Okay.,-0.165992,-10.1875,Okay.,-0.165992,-10.18750,Okay.,-0.165992,-10.1875,OK.,-0.808041,-11.0000,Okay.,-0.165992,-10.1875,ES2005d
1,"Okay, almost there.",Okay.,-0.202354,-10.1875,Okay. I want to say.,-0.761428,-5.21875,Okay.,-0.202354,-10.1875,Okay.,-0.202354,-10.1875,Okay. I was there.,-0.371616,-5.5000,ES2005d
2,Okay.,Okay,-0.634828,-17.0000,Ok,-2.403044,-19.62500,Okay.,-0.134573,-10.1875,OK.,-0.716196,-11.0000,Okay.,-0.134573,-10.1875,ES2005d


In [4]:
import re
import string

def normalize_text(text: str) -> str:
    """Mirrors the jiwer text normalization pipeline, except for last stage"""
    if not text:
        return ""
    
    # 1. jiwer.ToLowerCase()
    text = text.lower()
    
    # 2. jiwer.RemovePunctuation()
    # Removes standard punctuation: !"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # 3. jiwer.RemoveMultipleSpaces()
    text = re.sub(r'\s+', ' ', text)
    
    # 4. jiwer.Strip()
    text = text.strip()
    
    return text

In [5]:
def process_columns(df):
    llm_logprob_columns = [f'beam_{i}_llmlogprob' for i in range(1,6)]
    asr_logprob_columns = [f'beam_{i}_asrlogprob' for i in range(1,6)]
    
    llm_logprobs = torch.tensor(df[llm_logprob_columns].values)
    asr_logprobs = torch.tensor(df[asr_logprob_columns].values)
    
    # # ALL BEAMS
    # scores = F.softmax(ALPHA*llm_logprobs + 0.0asr_logprobs, dim=1)
    # highest_score_idxs = torch.argmax(scores, dim=1, keepdim=True)
    # highest_scores = torch.gather(scores, dim=1, index=highest_score_idxs)

    # JUST BEAM 1
    scores = llm_logprobs
    highest_scores = scores[torch.arange(scores.size(0)), torch.zeros(scores.size(0), dtype=torch.long)]


    # AGGREGATED STATS
    df['max_llmlogprobs'] = torch.max(llm_logprobs, dim=1).values.numpy()
    df['min_llmlogprobs'] = torch.min(llm_logprobs, dim=1).values.numpy()
    df['spread_llmlogprobs'] = df['max_llmlogprobs'] - df['min_llmlogprobs']

    df['max_asrlogprobs'] = torch.max(asr_logprobs, dim=1).values.numpy()
    df['min_asrlogprobs'] = torch.min(asr_logprobs, dim=1).values.numpy()
    df['spread_asrlogprobs'] = df['max_asrlogprobs'] - df['min_asrlogprobs']

    df['highest_score'] = highest_scores.numpy()
    
    df['text'] = df['beam_1_text']
    df['num_tokens_text'] = df['beam_1_text'].apply(
        lambda x: len(tokenizer.encode(x))
    )
    df['num_tokens_gt'] = df['gt'].apply(
        lambda x: len(tokenizer.encode(x))
    )
    
    
    df["rougeL"] = [
        rouge_l(normalize_text(pred), normalize_text(ref)).fmeasure
        for pred, ref in zip(df["text"], df["gt"])
    ]
    df["wer"] = [
        min(1.0, wer(reference = ref, hypothesis = pred, reference_transform = JIWER_TRANSFORM, hypothesis_transform = JIWER_TRANSFORM))
        for pred, ref in zip(df["text"], df["gt"])
    ]

process_columns(df)
process_columns(df_val)

In [6]:
df.head(5)
print(df.shape)

(88717, 29)


# Find Fillers in the dataset

In [10]:
from collections import Counter

def discover_fillers(df: pd.DataFrame, text_column: str, top_n: int = 100):
    """Finds the most common short utterances in your dataset."""
    short_utterances = []
    
    for text in df[text_column]:
        if not isinstance(text, str):
            continue
            
        # Clean the text
        clean_text = re.sub(r'[^\w\s]', '', text.lower()).strip()
        word_count = len(clean_text.split())
        
        # Look for 1 - 5 word utterances
        if 1 <= word_count <= 5:
            short_utterances.append(clean_text)
            
    # Return the most common ones
    return Counter(short_utterances).most_common(top_n)

# Example usage:
discovered_fillers = discover_fillers(df, 'text')
print(discovered_fillers)

(88717, 29)
[('you', 15998), ('yeah', 1629), ('okay', 1185), ('thank you', 1072), ('so', 313), ('and', 272), ('the', 226), ('yes', 217), ('um', 190), ('i', 190), ('ok', 188), ('yeah yeah', 140), ('right', 137), ('all right', 132), ('thanks', 130), ('no', 114), ('oh', 106), ('bye', 104), ('i dont know', 100), ('thats it', 83), ('good', 80), ('hmm', 79), ('im sorry', 72), ('to', 70), ('thank you very much', 70), ('i think', 68), ('well', 65), ('but', 63), ('alright', 62), ('that', 61), ('sorry', 58), ('it', 53), ('thanks for watching', 53), ('yeah okay', 52), ('for', 49), ('oh yeah', 49), ('you know', 49), ('uh', 47), ('thats right', 44), ('mmhmm', 42), ('one', 40), ('thats', 40), ('this', 36), ('of', 35), ('a', 35), ('yep', 32), ('great', 32), ('its', 32), ('ah', 31), ('exactly', 30), ('okay so', 29), ('okay okay', 29), ('what', 29), ('haha', 29), ('so yeah', 28), ('maybe', 27), ('now', 27), ('no no', 26), ('on', 26), ('thank you for watching', 25), ('cool', 24), ('we', 24), ('thats tru

# Stop Word Removal Analysis

In [ ]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords

# Download standard NLTK stopwords if you haven't already
nltk.download('stopwords', quiet=True)

# 1. Define your complete set of stop words
standard_stops = set(stopwords.words('english'))

# Add ASR-specific conversational fillers (no punctuation, all lowercase)
asr_fillers = {
    "yeah", "alright", "good", "mmhmm", "mm", "hmm", 
    "uh", "um", "ah", "okay", "ok", "oh"
}
all_stop_words = standard_stops.union(asr_fillers)

def contains_meaningful_text(text: str) -> bool:
    """
    Returns True if the text contains non-stop words after cleaning.
    """
    if not isinstance(text, str) or not text.strip():
        return False
        
    # Lowercase the text
    text = text.lower()
    
    # Remove punctuation. 
    # Note: "Mm-hmm" becomes "mmhmm", which is why "mmhmm" is in our custom list above.
    text = re.sub(r'[^\w\s]', '', text)
    
    # Split into words and check against stop words
    words = text.split()
    meaningful_words = [w for w in words if w not in all_stop_words]
    
    # If the list is not empty, there is meaningful text left
    return len(meaningful_words) > 0

# --- Example Usage ---

# Sample DataFrame based on your description
data = {
    'generated text': ["Yeah yeah yeah", "Hello world", "Alright, good", "The model works"],
    'ground truth text': ["yeah", "hi world", "mm-hmm", "model works fine"],
    'ROUGE-L': [0.5, 0.8, 0.0, 0.9],
    'WER': [0.2, 0.3, 1.0, 0.1]
}
test_df = pd.DataFrame(data)

# 2. Create boolean masks for both text columns
mask_gen = test_df['generated text'].apply(contains_meaningful_text)
mask_gt = test_df['ground truth text'].apply(contains_meaningful_text)

# 3. Filter the DataFrame to keep rows where BOTH texts are meaningful
# (If you want to drop rows where AT LEAST ONE is empty, use the `&` operator)
df_filtered = test_df[mask_gen & mask_gt].copy()

print("Original DataFrame:")
print(test_df)
print("\nFiltered DataFrame:")
print(df_filtered)